# EXP_030D — Text Ablation: Best Image + ViSoBERT + Concat + MSE
**Phase 3 | Text Backbone Ablation**
Research question: Does ViSoBERT (Vietnamese social media pretraining) improve over XLM-R for Foody reviews?
- Text model: `uitnlp/visobert` | Image model: Best from Phase 2 (set below)
- Fusion: Concatenation + MLP | Loss: MSE | Seed: 42 | AMP: enabled
> ⚠️ Prerequisite: Phase 2 must be completed. Set BEST_IMAGE_MODEL in STEP 4.

### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### STEP 2: Clone source code and install dependencies

In [ ]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt -q

### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data                                                                                                            
!cp /content/drive/MyDrive/SE365/data.zip ./data.zip                                                                      
!unzip -q data.zip                                                                                                        
!rm data.zip                                                                                                              
!ls -la ./data   

### STEP 4: Configure paths — ✏️ Set Phase 2 winner here

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/SE365'  # ✏️ Change if needed
EXP_ID = 'EXP_030D_bestimage_visobert_concat_mse'

# ✏️ SET based on Phase 2 results (same as EXP_030B)
BEST_IMAGE_MODEL  = 'swin_base_patch4_window7_224'
BEST_IMAGE_EXP_ID = 'EXP_020B_swinb_xlmr_concat_mse'

DRIVE_EXP_PATH = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
os.makedirs(DRIVE_EXP_PATH, exist_ok=True)
print(f'Artifacts will be saved to: {DRIVE_EXP_PATH}')
print(f'Using image backbone  : {BEST_IMAGE_MODEL}')
print(f'Loading image weights : {BEST_IMAGE_EXP_ID}')

### STEP 5: Load image weights from Phase 2 winner


In [ ]:
import os, shutil
os.makedirs('./checkpoints', exist_ok=True)
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_IMAGE_EXP_ID}/best_model_train_image.pth', './checkpoints/best_model_train_image.pth')
print(f'Loaded image weights from {BEST_IMAGE_EXP_ID}')


### STEP 6: Pre-train Text Branch
Fine-tune the text model before fusion to align its features with the score prediction task.

In [ ]:
!python main.py \
  --mode train_text \
  --text_model_name uitnlp/visobert \
  --epochs 20 \
  --batch_size 16 \
  --lr 1e-5 \
  --loss_fn mse \
  --seed 42 \
  --use_amp \
  --exp_dir ./experiments

!cp ./checkpoints/best_model_train_text.pth ./experiments/$EXP_ID/

### STEP 7: Train Fusion Model


In [ ]:
!python main.py \
  --mode train_fusion \
  --text_model_name uitnlp/visobert \
  --image_model_name {BEST_IMAGE_MODEL} \
  --epochs 15 \
  --batch_size 16 \
  --lr 1e-5 \
  --grad_accum_steps 2 \
  --patience 5 \
  --loss_fn mse \
  --unfreeze_text_layers 1 \
  --unfreeze_image_layers 1 \
  --seed 42 \
  --use_amp \
  --exp_id EXP_030D_bestimage_visobert_concat_mse \
  --exp_dir ./experiments

### STEP 8: Save to Drive + print metrics


In [ ]:
import json
!cp -r ./experiments/$EXP_ID/* $DRIVE_EXP_PATH/

with open(f'./experiments/{EXP_ID}/metrics.json') as f:
    m = json.load(f)

print(f'\n=== {EXP_ID} Results ===')
print(f"Loss (val)   : {m['loss']:.4f}")
print()
print("             MAE      RMSE      R2")
print(f"  food     : {m['mae_food']:.4f}   {m['rmse_food']:.4f}   {m['r2_food']:.4f}")
print(f"  price    : {m['mae_price']:.4f}   {m['rmse_price']:.4f}   {m['r2_price']:.4f}")
print(f"  atmos    : {m['mae_atmos']:.4f}   {m['rmse_atmos']:.4f}   {m['r2_atmos']:.4f}")
print(f"  service  : {m['mae_service']:.4f}   {m['rmse_service']:.4f}   {m['r2_service']:.4f}")
print(f"  overall  : {m['mae_overall']:.4f}   {m['rmse_overall']:.4f}   {m['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {m['mean_mae']:.4f}")
print(f"  aspect_mae : {m['aspect_mae']:.4f}")
print(f"  overall_mae: {m['overall_mae']:.4f}")